In [4]:
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import scipy.stats as stats
import math

### Part 1

In [5]:
def monte_carlo(function, sample_size):
    U = np.random.uniform(0, 1, sample_size)
    X = function(U)
    estimate = np.mean(X)

    conf_int = stats.t.interval(confidence=0.95,df=sample_size - 1,loc=estimate,scale=np.std(X, ddof=1) / np.sqrt(sample_size))

    return estimate, conf_int


In [6]:
sample_size = 100

estimate, conf_int = monte_carlo(np.exp, sample_size)

print("point estimate:", estimate)
print("confidence interval:", conf_int)
print("exact value:", np.e - 1)


point estimate: 1.5836175212083798
confidence interval: (np.float64(1.4912763346968678), np.float64(1.6759587077198919))
exact value: 1.718281828459045


### Part 2

In [7]:
def anti_monte_carlo(function, sample_size):
    U = np.random.uniform(0, 1, sample_size)

    Y = (function(U) + function(1 - U)) / 2

    estimate = np.mean(Y)

    conf_int = stats.t.interval(confidence=0.95,df=sample_size - 1,loc=estimate,scale=np.std(Y, ddof=1) / np.sqrt(sample_size))

    return estimate, conf_int

In [8]:
estimate, conf_int = anti_monte_carlo(np.exp, sample_size)

print("point estimate:", estimate)
print("confidence interval:", conf_int)
print("exact value:", np.e - 1)

point estimate: 1.7166902295269006
confidence interval: (np.float64(1.704722963740091), np.float64(1.7286574953137104))
exact value: 1.718281828459045


### Part 3

In [9]:


def control_variate_monte_carlo(function, sample_size):
    U = np.random.uniform(0, 1, sample_size)
    X = function(U)

    c = -np.cov(X, U, ddof=1)[0, 1] / np.var(U, ddof=1)

    Y = X + c * (U - 1/2)

    estimate = np.mean(Y)

    conf_int = stats.t.interval(confidence=0.95,df=sample_size - 1,loc=estimate,scale=np.std(Y, ddof=1) / np.sqrt(sample_size))

    return estimate, conf_int, c


estimate, conf_int, c = control_variate_monte_carlo(np.exp, sample_size)

print("c:", c)
print("point estimate:", estimate)
print("confidence interval:", conf_int)
print("exact value:", np.e - 1)


c: -1.716464802654706
point estimate: 1.7115877710838245
confidence interval: (np.float64(1.6998529557810926), np.float64(1.7233225863865564))
exact value: 1.718281828459045


### part 4

In [10]:
def stratified_sampling(sample_size, strats=10):
    reps = sample_size // strats
    Y = []

    for i in range(reps):
        total = 0

        for j in range(strats):
            U = np.random.uniform(j / strats, (j + 1) / strats)
            total += np.exp(U)

        Y.append(total / strats)

    estimate = np.mean(Y)

    conf_int = stats.t.interval(
        confidence=0.95,
        df=reps - 1,
        loc=estimate,
        scale=np.std(Y, ddof=1) / np.sqrt(reps)
    )

    return estimate, conf_int

In [20]:
sample_size = 100

estimate, conf_int = stratified_sampling(sample_size, strats=10)

print("point estimate:", estimate)
print("confidence interval:", conf_int)
print("exact value:", np.e - 1)

point estimate: 1.7159968065136284
confidence interval: (np.float64(1.7072144417632056), np.float64(1.7247791712640512))
exact value: 1.718281828459045


### Part 5

#### Parameters

In [38]:
m = 10
mean_service_time = 8
mean_interarrival_time = 1
t_service_rate = 1/mean_service_time
interArrival_rate = 1/mean_interarrival_time

n_repetitions = 10
n_customers = 10000

In [39]:
def blocking(mean_arrival_time, mean_service_time, customers, units):
    t_interArrival = np.random.exponential(mean_arrival_time, customers)
    t_arrival = np.cumsum(t_interArrival)
    t_service = np.random.exponential(mean_service_time, customers)

    service_units = [0] * units
    n_blocked = 0

    for i in range(customers):
        arrival_time = t_arrival[i]
        t_departure = arrival_time + t_service[i]

        vacancies = [x for x in service_units if x <= arrival_time]

        if len(vacancies) > 0:
            indx = service_units.index(vacancies[0])
            service_units[indx] = t_departure
        else:
            n_blocked += 1

    blocked_fraction = n_blocked / customers
    average_interarrival_time = np.mean(t_interArrival)

    return blocked_fraction, average_interarrival_time

In [40]:
X = []
Y = []

for i in range(n_repetitions):
    blocked_fraction, average_interarrival_time = blocking(
        mean_interarrival_time,
        mean_service_time,
        n_customers,
        m
    )

    X.append(blocked_fraction)
    Y.append(average_interarrival_time)

X = np.array(X)
Y = np.array(Y)

mu_Y = mean_interarrival_time

c = -np.cov(X, Y, ddof=1)[0, 1] / np.var(Y, ddof=1)

X_cv = X + c * (Y - mu_Y)

print("ordinary estimate:", np.mean(X))
print("control variate estimate:", np.mean(X_cv))

print("ordinary variance:", np.var(X, ddof=1))
print("control variate variance:", np.var(X_cv, ddof=1))

print("c:", c)
print("mean Y:", np.mean(Y))
print("variance Y:", np.var(Y, ddof=1))

ordinary estimate: 0.12025
control variate estimate: 0.11978706095014391
ordinary variance: 1.1922777777777767e-05
control variate variance: 7.658675738197745e-06
c: 0.21673506836275216
mean Y: 0.9978640325566455
variance Y: 9.077561805777153e-05


### Part 7

In [15]:
def h(a):
    z = np.random.normal(0,1)
    return int(z > a)


In [ ]:
def important_monte_carlo(a, sample_size,std = 1):
    Y = np.random.normal(a,std,sample_size)
    total = 0
    for y in Y:
        
        f = stats.norm.pdf(y,0,1)
        g = stats.norm.pdf(y,a,std)
        total += h(y) * (f/g)
    
    result = total/sample_size
    return result


### crude normal tail

In [ ]:
def crude_normal_tail(a, sample_size):
    Z = np.random.normal(0, 1, sample_size)

    X = Z > a

    estimate = np.mean(X)

    conf_int = stats.t.interval(confidence=0.95,df=sample_size - 1,loc=estimate,scale=np.std(X, ddof=1) / np.sqrt(sample_size))

    return estimate, conf_int

### IS-Tail

In [ ]:
def importance_sampling_normal_tail(a, sample_size, std=1):
    Y = np.random.normal(a, std, sample_size)

    f = stats.norm.pdf(Y, loc=0, scale=1)
    g = stats.norm.pdf(Y, loc=a, scale=std)

    X = (Y > a) * (f / g)

    estimate = np.mean(X)

    conf_int = stats.t.interval(confidence=0.95,df=sample_size - 1,loc=estimate,scale=np.std(X, ddof=1) / np.sqrt(sample_size))

    return estimate, conf_int

### Theoretical values

In [43]:
print("a = 2", 1 - stats.norm.cdf(2))
print("a = 4", 1 - stats.norm.cdf(4))

0.02275013194817921
3.167124183311998e-05


### crude normal tail test

In [27]:
sample_size_tests = [1000,10000,100000]
a_arr = [2,4]
print("---------- Crude normal tail ----------")
for a in a_arr:
     for sample_size_test in sample_size_tests:
          estimate, conf_int = crude_normal_tail(a, sample_size_test)
          print("a:", a, "\nSample size:", sample_size_test ,"\nEstimate:", estimate, "\nConfidence interval:", conf_int)


---------- Crude normal tail ----------
a: 2 
Sample size: 1000 
Estimate: 0.019 
Confidence interval: (np.float64(0.010523762050666732), np.float64(0.027476237949333265))
a: 2 
Sample size: 10000 
Estimate: 0.0227 
Confidence interval: (np.float64(0.019780225854317046), np.float64(0.025619774145682957))
a: 2 
Sample size: 100000 
Estimate: 0.02247 
Confidence interval: (np.float64(0.021551409176428688), np.float64(0.023388590823571313))
a: 4 
Sample size: 1000 
Estimate: 0.0 
Confidence interval: (np.float64(nan), np.float64(nan))
a: 4 
Sample size: 10000 
Estimate: 0.0 
Confidence interval: (np.float64(nan), np.float64(nan))
a: 4 
Sample size: 100000 
Estimate: 5e-05 
Confidence interval: (np.float64(6.174219054262663e-06), np.float64(9.382578094573735e-05))


c:\Users\kelvi\AppData\Local\Programs\Python\Python313\Lib\site-packages\scipy\stats\_distn_infrastructure.py:2323: RuntimeWarning: invalid value encountered in multiply
  lower_bound = _a * scale + loc
c:\Users\kelvi\AppData\Local\Programs\Python\Python313\Lib\site-packages\scipy\stats\_distn_infrastructure.py:2324: RuntimeWarning: invalid value encountered in multiply
  upper_bound = _b * scale + loc


### Importance sampling tests

In [42]:
print("---------- Importance sampling tail ----------")
for a in a_arr:
     for sample_size_test in sample_size_tests:
          estimate, conf_int = importance_sampling_normal_tail(a, sample_size_test)
          print("a:", a, "\nSample size:", sample_size_test ,"\nEstimate:", estimate, "\nConfidence interval:", conf_int)

---------- Importance sampling tail ----------
a: 2 
Sample size: 1000 
Estimate: 0.02376972001757828 
Confidence interval: (np.float64(0.021523624540591337), np.float64(0.026015815494565223))
a: 2 
Sample size: 10000 
Estimate: 0.022795331496168757 
Confidence interval: (np.float64(0.022111211881023298), np.float64(0.023479451111314217))
a: 2 
Sample size: 100000 
Estimate: 0.022861042755297663 
Confidence interval: (np.float64(0.022644390098509274), np.float64(0.02307769541208605))
a: 4 
Sample size: 1000 
Estimate: 2.9801285442951358e-05 
Confidence interval: (np.float64(2.5709687410859528e-05), np.float64(3.389288347504319e-05))
a: 4 
Sample size: 10000 
Estimate: 3.189686102619142e-05 
Confidence interval: (np.float64(3.057338598005415e-05), np.float64(3.322033607232869e-05))
a: 4 
Sample size: 100000 
Estimate: 3.158118093399747e-05 
Confidence interval: (np.float64(3.11646258448695e-05), np.float64(3.1997736023125435e-05))


In [17]:
print(important_monte_carlo(4,10000))

0.15240763756297207


### Part 8

In [45]:
def importance_sampling_integral(lambda_value, sample_size):
    Y = np.random.exponential(scale=1/lambda_value, size=sample_size)

    X = (np.exp((1 + lambda_value) * Y) / lambda_value) * (Y <= 1)

    estimate = np.mean(X)

    conf_int = stats.t.interval(confidence=0.95,df=sample_size - 1,loc=estimate,scale=np.std(X, ddof=1) / np.sqrt(sample_size))

    variance = np.var(X, ddof=1)

    return estimate, conf_int, variance



In [46]:
sample_size = 100000

for lambda_value in [0.5, 1, 1.35, 1.5, 2, 3]:
    estimate, conf_int, variance = importance_sampling_integral(lambda_value, sample_size)

    print("lambda:", lambda_value)
    print("estimate:", estimate)
    print("confidence interval:", conf_int)
    print("variance:", variance)
    print()

lambda: 0.5
estimate: 1.7097353719355728
confidence interval: (np.float64(1.6946022716454625), np.float64(1.724868472225683))
variance: 5.96141183152003

lambda: 1
estimate: 1.719879248066555
confidence interval: (np.float64(1.7084493541022123), np.float64(1.7313091420308977))
variance: 3.400773498330463

lambda: 1.35
estimate: 1.7226368051798495
confidence interval: (np.float64(1.7116579427319871), np.float64(1.733615667627712))
variance: 3.1376752537986903

lambda: 1.5
estimate: 1.718519833701033
confidence interval: (np.float64(1.7075116717084913), np.float64(1.7295279956935747))
variance: 3.154444773931285

lambda: 2
estimate: 1.7249696631837064
confidence interval: (np.float64(1.712926703699744), np.float64(1.7370126226676688))
variance: 3.77537204055199

lambda: 3
estimate: 1.719836420180305
confidence interval: (np.float64(1.703486704340653), np.float64(1.7361861360199569))
variance: 6.958469413851629

